In [3]:
from tensorflow.keras.models import load_model
import pandas as pd
import pickle

In [4]:
modelO = load_model('model.h5')

In [5]:
input = pd.DataFrame({
    'CreditScore': [600],
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : [80],
    'Tenure' : [3],
    'Balance' : [60000],
    'NumOfProducts' : [2],
    'HasCrCard' : [1],
    'IsActiveMember' : [1],
    'EstimatedSalary' : [50000]
})

In [6]:
input

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,80,3,60000,2,1,1,50000


In [7]:
with open('label_encoder.pkl', 'rb') as f:
    lc = pickle.load(f)
with open('onehot_encoder.pkl', 'rb') as f:
    ohe = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)        

In [8]:
input['Gender'] = lc.transform(input['Gender'])

In [9]:
input

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,80,3,60000,2,1,1,50000


In [10]:
ohegeo = ohe.transform(input[['Geography']])
print(ohegeo)
ohegeo[0][0], ohegeo[0][1], ohegeo[0][2]

[[1. 0. 0.]]


(np.float64(1.0), np.float64(0.0), np.float64(0.0))

In [11]:
geo =  pd.DataFrame(ohegeo, columns=ohe.get_feature_names_out(['Geography']))

In [12]:
geo

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [13]:
input_ready = pd.concat([input, geo], axis=1)
input_ready.drop('Geography', axis=1, inplace=True)
input_ready

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,80,3,60000,2,1,1,50000,1.0,0.0,0.0


In [14]:
scaled_input = scaler.transform(input_ready)

In [15]:
scaled_input

array([[-0.54204762,  0.91055421,  3.9121348 , -0.68894811, -0.26230046,
         0.81966266,  0.64598061,  0.97071435, -0.88153859,  0.99828718,
        -0.57559072, -0.57779016]])

In [16]:
Prediction = modelO.predict(scaled_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step


In [17]:
Prediction

array([[0.00510118]], dtype=float32)

In [18]:
Prediction_probability = Prediction[0][0]
Prediction_probability = float(round(Prediction_probability, 2)*100)
Prediction_probability

1.0

In [19]:
if Prediction_probability > 50:
    print(f"The customer is likely to churn with a probability of {Prediction_probability}%")
else:
    print(f"The customer is not likely to churn with a probability of {Prediction_probability}%")    

The customer is not likely to churn with a probability of 1.0%
